In [1]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("VADER_tokenize_score_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [2]:
# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU TỪ ICEBERG
# ==========================================
table_tokens = "my_catalog.processed_zone_finnhub.news_tokens_sentiment_score"
table_lemmas = "my_catalog.processed_zone_finnhub.news_lemmas_sentiment_score"

df_tokens = spark.read.table(table_tokens)
df_lemmas = spark.read.table(table_lemmas)

In [10]:
# ==========================================
# BƯỚC 2: CHỌN CỘT VÀ LOẠI BỎ TRÙNG LẶP (DEDUPLICATE)
# ==========================================
# Sử dụng .dropDuplicates([JOIN_KEY]) để đảm bảo mỗi ID chỉ có 1 dòng duy nhất

df_tok_selected = df_tokens.select(
    col(JOIN_KEY),
    col("title_vader_score").alias("tok_score"),
    col("title_vader_sentiment").alias("tok_label")
).dropDuplicates([JOIN_KEY]) # <--- Thêm hàm này

df_lem_selected = df_lemmas.select(
    col(JOIN_KEY),
    col("title_vader_score").alias("lem_score"),
    col("title_vader_sentiment").alias("lem_label")
).dropDuplicates([JOIN_KEY]) # <--- Thêm hàm này

In [11]:
# ==========================================
# BƯỚC 3: KẾT NỐI (JOIN) 2 DATAFRAME
# ==========================================
df_compare = df_tok_selected.join(df_lem_selected, on=JOIN_KEY, how="inner")

In [12]:
# ==========================================
# BƯỚC 4: THÊM CÁC CỘT SO SÁNH PHÂN TÍCH
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, abs as spark_abs, round as spark_round

df_analysis = df_compare \
    .withColumn(
        "score_diff", 
        spark_round(spark_abs(col("tok_score") - col("lem_score")), 4)
    ) \
    .withColumn(
        "is_label_match", 
        when(col("tok_label") == col("lem_label"), "Khớp 🟢").otherwise("Lệch 🔴")
    )

In [16]:
# ==========================================
# BƯỚC 5: HIỂN THỊ VÀ THỐNG KÊ
# ==========================================
print("🔍 1. Hiển thị mẫu so sánh kết quả (Cả trùng và lệch):")
# Bỏ hàm filter để xem dữ liệu tổng quát, sắp xếp theo độ chênh lệch điểm để dễ quan sát
df_analysis.orderBy(col("score_diff").desc()) \
    .show(200, truncate=False)

print("📊 2. Thống kê tổng quan mức độ khớp nhãn:")
# Xem tổng số lượng dòng khớp và lệch
df_analysis.groupBy("is_label_match").count().show()

print("📈 3. Xem độ chênh lệch điểm số trung bình:")
# Xem trung bình điểm số thay đổi như thế nào giữa 2 phương pháp
df_analysis.selectExpr("avg(score_diff) as average_score_difference").show()

🔍 1. Hiển thị mẫu so sánh kết quả (Cả trùng và lệch):
+---------+---------+------------+---------+------------+----------+--------------+
|id       |tok_score|tok_label   |lem_score|lem_label   |score_diff|is_label_match|
+---------+---------+------------+---------+------------+----------+--------------+
|139055338|0.4215   |Tích cực 🟢 |-0.8689  |Tiêu cực 🔴 |1.2904    |Lệch 🔴       |
|136612925|0.5859   |Tích cực 🟢 |-0.5106  |Tiêu cực 🔴 |1.0965    |Lệch 🔴       |
|140273310|-0.1531  |Tiêu cực 🔴 |0.8442   |Tích cực 🟢 |0.9973    |Lệch 🔴       |
|140122365|0.2732   |Tích cực 🟢 |-0.6808  |Tiêu cực 🔴 |0.954     |Lệch 🔴       |
|136063091|0.2732   |Tích cực 🟢 |-0.6808  |Tiêu cực 🔴 |0.954     |Lệch 🔴       |
|139233497|0.2732   |Tích cực 🟢 |-0.6808  |Tiêu cực 🔴 |0.954     |Lệch 🔴       |
|136647797|0.4404   |Tích cực 🟢 |-0.5106  |Tiêu cực 🔴 |0.951     |Lệch 🔴       |
|138047518|-0.5106  |Tiêu cực 🔴 |0.4215   |Tích cực 🟢 |0.9321    |Lệch 🔴       |
|137874033|0.5106   |Tích cực 🟢 |-0.4019  |Tiê